# Metrics

In [1]:
from google.cloud import bigquery
import gender_guesser.detector as gender
import pandas as pd

from tqdm import tqdm
tqdm.pandas()

import warnings
warnings.filterwarnings("ignore")

## Class metrics

In [2]:
class MetricsComputation:
    
    def __init__(self, inst_ids_list, PROJECT_ID, DATASET_ID, cerca_centers):
        self.inst_ids = list(set(sum(list(inst_ids_list.values()), [])))
        self.GBQ_PROJECT_ID = PROJECT_ID
        self.GBQ_DATASET_ID = DATASET_ID
        self.gend = gender.Detector()
        self.cerca_centers = cerca_centers
        
    def bg_query(self, query):
        client = bigquery.Client(project = self.GBQ_PROJECT_ID)
        df = client.query(query)
        return df.to_dataframe()
    
    def gender_detection(self):
        df_gender = self.df_inst.drop_duplicates(subset = ['display_name']).reset_index(drop = True)
        df_gender['gender'] = df_gender.first_name.progress_apply(lambda x: self.gend.get_gender(x))
        self.df_inst = self.df_inst.merge(df_gender[['display_name', 'gender']], on = 'display_name', how = 'left')
        return self.df_inst
        
    def get_institution_data(self):
        institutions_sql = "(" + ",".join(f"'{i}'" for i in self.inst_ids) + ")"

        sql = f"""SELECT ww.DOI, a.display_name, wa.author_order, wa.INSTITUTION_ID, wins.COUNTRY_CODE
                    FROM `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works` ww
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID AND CAST(wa.INSTITUTION_ID AS STRING) IN {institutions_sql}
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.authors` a ON a.ID = wa.author_id
                    LEFT JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
                    WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024"""
                    
        self.df_inst = self.bg_query(sql).dropna(subset = ['DOI']).reset_index(drop = True)
        self.df_inst['first_name'] = self.df_inst['display_name'].str.split(' ').str[0]
        self.df_inst['gender'] = self.df_inst.first_name.progress_apply(lambda x: self.gend.get_gender(x))
        
        return self.df_inst
    
    def get_institution_collaboration(self):
        in_query = str(tuple(self.df_inst.DOI.unique().tolist()))
        in_query = in_query.replace(',)', ')')

        sql = f"""SELECT ww.DOI, wa.INSTITUTION_ID, wins.COUNTRY_CODE
                    FROM `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works` ww
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
                    WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024 AND ww.DOI IN {in_query}
                """
        self.df_inst_colab = self.bg_query(sql).drop_duplicates().reset_index(drop = True)
        return self.df_inst_colab
        
    def metrics(self):
        
        self.df_inst = self.get_institution_data()
        
        pubs_number = self.df_inst.DOI.nunique()
        
        df_led_pubs = self.df_inst[self.df_inst.author_order == 1]
        per_leds_pubs = df_led_pubs.DOI.nunique() / pubs_number

        df_fem_pubs = self.df_inst[self.df_inst.gender == 'female']
        per_fem_pub = df_fem_pubs.DOI.nunique() / pubs_number
        
        df_led_fem_pubs = df_fem_pubs[df_fem_pubs.author_order == 1]
        per_led_fem_pubs = df_led_fem_pubs.DOI.nunique() / pubs_number
        
        self.df_inst_colab = self.get_institution_collaboration()

        cerca_inst = list(set(sum(list(self.cerca_centers.values()), [])))
        cerca_inst = [int(x) for x in cerca_inst if isinstance(x, (str, int)) and str(x).isdigit()]
        df_cerca = self.df_inst_colab[(self.df_inst_colab.INSTITUTION_ID.isin(cerca_inst)) & (~self.df_inst_colab.INSTITUTION_ID.isin([int(x) for x in self.inst_ids]))]
        per_cerca_pubs = df_cerca.DOI.nunique() / pubs_number
        
        df_inter = self.df_inst_colab[self.df_inst_colab.COUNTRY_CODE != self.df_inst.COUNTRY_CODE.dropna().unique()[0]]
        per_inter_pubs = df_inter.DOI.nunique() / pubs_number

        return pubs_number, per_leds_pubs, per_fem_pub, per_led_fem_pubs, per_cerca_pubs, per_inter_pubs
        

**It missed the Beta and the MESA+ and the LSTM. For the first one we will use previous results; for the other one we use raw affiliation string search in the parent affiliation**

In [ ]:
gbq_project = 'siris-datasets'
gbq_dataset = 'openalex'

cerca_centers = {# 'BETA' : [''], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

interest_centers = {'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109'],
                    'Centro de Investigación y Tecnología Agroalimentaria de Aragón (CITA)' : ['4210122787'],
                    'INAGRO - Research & advice in agriculture and horticulture' : ['4210109617'],
                    'KWR Water Research Institute' : ['4210139073'],
                    'THE JAMES HUTTON INSTITUTE' : ['15477984'],
                    'SUOMEN YMPARISTOKESKUS (SYKE)' : ['2800424308'],
                    'UK CENTRE FOR ECOLOGY & HYDROLOGY' : ['4210092773'],
                    'International Iberian Nanotechnology Laboratory (INL)' : ['4210141319'],
                    # 'MESA+ Institute (University of Twente.)' : [''], # NOT IN OA
                    'Imdea Nanociencia' : ['2802543619'],
                    'The Swiss Tropical and Public Health Institute (SwissTPH)' : ['158937107'],
                    'The London School of Hygiene and Tropical Medicine (LSHTM)' : ['4210089966'],
                    'The Liverpool School of Tropical Medicine (LSTM)' : ['94839184'],
                    'FUNDACION PARA LA INVESTIGACION BIOMEDICA DEL HOSPITAL UNIVERSITARIO 12 DE OCTUBRE' : ['4210086614'],
                    'CENTRE HOSPITALIER UNIVERSITAIRE DE TOULOUSE' : ['3019448017'],
                    'FONDAZIONE IRCCS CA\' GRANDA - OSPEDALE MAGGIORE POLICLINICO' : ['2803066834']}

cerca_institutions = list(set(sum(list(cerca_centers.values()), [])))
cerca_institutions = [int(x) for x in cerca_institutions if isinstance(x, (str, int)) and str(x).isdigit()]

results = []
for name, ids in tqdm(interest_centers.items(), total = len(interest_centers), desc = "Processing centers"):
    tqdm.write(f"Processing: {name}")
    center_dict = {name: ids}
    metrics_obj = MetricsComputation(center_dict, gbq_project, gbq_dataset, cerca_centers)
    results.append(metrics_obj.metrics())
df_results = pd.DataFrame(results, interest_centers.keys(), columns = ['# Pubs', '% led Pubs', '% Fem Pubs', '% Fem led Pubs', '% CERCA Colab Pubs', '% Inter Colab Pubs'])
df_results.to_csv('../data/processed/benchmark_results.csv')
df_results

Processing centers:   0%|          | 0/18 [00:00<?, ?it/s]

Processing: CREAF


Processing centers:   6%|▌         | 1/18 [00:04<01:20,  4.71s/it]

Processing: ICN2


Processing centers:  11%|█         | 2/18 [00:09<01:12,  4.53s/it]

...

Processing: CENTRE HOSPITALIER UNIVERSITAIRE DE TOULOUSE


Processing centers:  94%|█████████▍| 17/18 [02:09<00:11, 11.85s/it]

Processing: FONDAZIONE IRCCS CA' GRANDA - OSPEDALE MAGGIORE POLICLINICO


Processing centers: 100%|██████████| 18/18 [02:28<00:00,  8.27s/it]


,# Pubs,% led Pubs,% Fem Pubs,% Fem led Pubs,% CERCA Colab Pubs,% Inter Colab Pubs
CREAF,1315,0.310266,0.333840,0.098859,0.002281,0.822814
ICN2,1060,0.366981,0.345283,0.090566,0.013208,0.771698
ISGlobal,3308,0.347642,0.615780,0.191657,0.058041,0.815599
ResearchMar,2476,0.314620,0.632876,0.159935,0.075929,0.584006
Centro de Investigación y Tecnología Agroalimentaria de Aragón (CITA),1026,0.481481,0.547758,0.214425,0.007797,0.422027
INAGRO - Research & advice in agriculture and horticulture,82,0.231707,0.353659,0.036585,0.000000,0.487805
KWR Water Research Institute,531,0.333333,0.282486,0.092279,0.000000,0.647834
THE JAMES HUTTON INSTITUTE,1732,0.334296,0.426097,0.137413,0.005196,0.669169
SUOMEN YMPARISTOKESKUS (SYKE),1395,0.351254,0.656631,0.184229,0.004301,0.525448
UK CENTRE FOR ECOLOGY & HYDROLOGY,3015,0.356551,0.455390,0.105141,0.015257,0.660033


## Raw string search for the affiliations without id

In [16]:
def bg_query( query):
    client = bigquery.Client(project = gbq_project)
    df = client.query(query)
    return df.to_dataframe()

interest_centers = {'MESA+ Institute (University of Twente.)' : ['94624287']} # NOT IN OA - AFFILIATON PARENT

inst_ids = list(set(sum(list(interest_centers.values()), [])))
institutions_sql = "(" + ",".join(f"'{i}'" for i in inst_ids) + ")"

sql = f"""SELECT ww.DOI, a.display_name, wa.author_order, wa.INSTITUTION_ID, wins.COUNTRY_CODE, war.raw_affiliation
            FROM `{gbq_project}.{gbq_dataset}.works` ww
            JOIN `{gbq_project}.{gbq_dataset}.works_authorships` wa ON wa.WORK_ID = ww.ID AND CAST(wa.INSTITUTION_ID AS STRING) IN {institutions_sql}
            JOIN `{gbq_project}.{gbq_dataset}.authors` a ON a.ID = wa.author_id
            LEFT JOIN `{gbq_project}.{gbq_dataset}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
            LEFT JOIN `{gbq_project}.{gbq_dataset}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
            WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024"""
            
df_parent = bg_query(sql).dropna(subset = ['DOI']).reset_index(drop = True)
df_inst = df_parent[df_parent['raw_affiliation'].str.contains('mesa', case=False, na=False)].drop_duplicates().reset_index(drop=True)
df_inst

,DOI,display_name,author_order,INSTITUTION_ID,COUNTRY_CODE,raw_affiliation
0,10.1021/acsenergylett.3c00697,Geert Brocks,20,94624287,NL,"Computational Materials Science, Faculty of Sc..."
1,10.1021/acs.jpcc.0c10525,Uufuk Halisdemir,22,94624287,NL,Faculty of Science and Technology and MESA+ In...
2,10.1364/iprsn.2022.itu3b.1,Jelmer J. Renema,24,94624287,NL,"Mesa+ Institute for Nanotechnology, University..."
3,10.1016/j.joule.2024.06.017,Monica Morales‐Masis,35,94624287,NL,"MESA+ Institute for Nanotechnology, University..."
4,10.1038/s41467-022-33913-6,Mario Vretenar,62,94624287,NL,"Adaptive Quantum Optics (AQO), MESA+Institute ..."
...,...,...,...,...,...,...
4067,10.1038/s41467-023-44331-7,Guus Rijnders,18,94624287,NL,"MESA+ Institute for Nanotechnology, University..."
4068,10.21203/rs.3.rs-265347/v1,H.J.M. Bouwmeester,18,94624287,NL,"Electrochemistry Research group, Membrane Scie..."
4069,10.1364/quantum.2022.qtu3a.3,Jelmer J. Renema,18,94624287,NL,"MESA+ Institute for Nanotechnology, University..."
4070,10.1038/s41467-021-27898-x,Gertjan Koster,18,94624287,NL,"MESA+ Institute for Nanotechnology, University..."


In [ ]:
pubs_number = df_inst.DOI.nunique()
per_led_pubs = df_inst[df_inst.author_order == 1].DOI.nunique() / pubs_number

gend = gender.Detector()
df_fem_pubs = df_inst.dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS
df_fem_pubs['first_name'] = df_fem_pubs['display_name'].str.split(' ').str[0]
df_fem_pubs['gender'] = df_fem_pubs.first_name.progress_apply(lambda x: gend.get_gender(x))
df_fem_pubs = df_fem_pubs[df_fem_pubs.gender == 'female']
per_fem_pubs = df_fem_pubs.DOI.nunique() / pubs_number

per_led_fem_pubs = df_fem_pubs[df_fem_pubs.author_order == 1].DOI.nunique() / pubs_number

# COLAB 
in_query = str(tuple(df_inst.DOI.unique().tolist()))
in_query = in_query.replace(',)', ')')
sql = f"""SELECT ww.DOI, wa.INSTITUTION_ID, wins.COUNTRY_CODE
            FROM `{gbq_project}.{gbq_dataset}.works` ww
            JOIN `{gbq_project}.{gbq_dataset}.works_authorships` wa ON wa.WORK_ID = ww.ID
            JOIN `{gbq_project}.{gbq_dataset}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
            WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024 AND ww.DOI IN {in_query}"""
df_inst_colab = bg_query(sql).drop_duplicates().reset_index(drop = True)

cerca_inst = list(set(sum(list(cerca_centers.values()), [])))
cerca_inst = [int(x) for x in cerca_inst if isinstance(x, (str, int)) and str(x).isdigit()]

df_cerca = df_inst_colab[(df_inst_colab.INSTITUTION_ID.isin(cerca_inst)) & (~df_inst_colab.INSTITUTION_ID.isin([int(x) for x in inst_ids]))]
per_cerca_pubs = df_cerca.DOI.nunique() / pubs_number

df_inter = df_inst_colab[df_inst_colab.COUNTRY_CODE != df_inst.COUNTRY_CODE.dropna().unique()[0]]
per_inter_pubs = df_inter.DOI.nunique() / pubs_number

results = [pubs_number, per_led_pubs, per_fem_pubs, per_led_fem_pubs, per_cerca_pubs, per_inter_pubs]
df_results = pd.DataFrame([results], interest_centers.keys(), columns = ['# Pubs', '% led Pubs', '% Fem Pubs', '% Fem led Pubs', '% CERCA Colab Pubs', '% Inter Colab Pubs'])
df_results

100%|██████████| 4072/4072 [00:00<00:00, 210324.69it/s]


,# Pubs,% led Pubs,% Fem Pubs,% Fem led Pubs,% CERCA Colab Pubs,% Inter Colab Pubs
MESA+ Institute (University of Twente.),1466,0.521146,0.295362,0.087995,0.002046,0.643929
